# DS2002 · Pandas Challenge

**Lab — 2026-09-18 · Fall 2026**  

---

## Lab 04 — Pandas Challenge

Four hundred generated orders. Each question builds toward a demand report you could hand a vendor.

The data is seeded, so everyone's numbers should match. That is deliberate: if your total revenue differs from your neighbor's, one of you has a bug, and the assertions at the end will tell you which.

Every answer needs the number **and** a sentence saying what it means. A cell that prints `4218.5` with no interpretation is half an answer.

In [1]:
import pandas as pd, numpy as np
rng = np.random.default_rng(4)
n = 400
df = pd.DataFrame({
    'vendor_id': rng.choice(['V-01','V-05','V-10','V-18'], n),
    'category': rng.choice(['Food','Merch','RainGear','Drink'], n, p=[.5,.2,.1,.2]),
    'qty': rng.integers(1, 4, n),
    'price': rng.choice([4.5, 6.0, 7.5, 12.0, 24.0], n),
})
df.head()

,vendor_id,category,qty,price
0,V-10,Drink,2,24.0
1,V-18,RainGear,1,12.0
2,V-18,Drink,3,4.5
3,V-10,Food,2,12.0
4,V-18,Drink,3,7.5


### Q1 — Add `revenue`, then report total revenue and total units.

*Expected: 400 rows, and revenue should land between $8,000 and $9,000.*

In [2]:
# TODO
df['revenue'] = df['qty'] * df['price']
print("total revenue: ", df['revenue'].sum())
print("total units: ", df['qty'].sum())
df
# This means the shop moved ~783 units of inventory
# and pulled in $8,520 across all 400 orders.

total revenue:  8520.0
total units:  783


,vendor_id,category,qty,price,revenue
0,V-10,Drink,2,24.0,48.0
1,V-18,RainGear,1,12.0,12.0
2,V-18,Drink,3,4.5,13.5
3,V-10,Food,2,12.0,24.0
4,V-18,Drink,3,7.5,22.5
...,...,...,...,...,...
395,V-18,Merch,1,12.0,12.0
396,V-01,Merch,2,24.0,48.0
397,V-10,Food,3,7.5,22.5
398,V-18,Merch,2,24.0,48.0


### Q2 — Revenue by category, highest to lowest.

Include the share of total as a percentage in the same table.

In [3]:
# TODO
by_category = df.groupby('category')['revenue'].sum().to_frame()
by_category['share'] = by_category['revenue'] / by_category['revenue'].sum()
by_category.sort_values('revenue', ascending=False)
# This means Food drives around half of all revenue (50.4%),
# while RainGear is the smallest category at 10.6%.

,revenue,share
category,,
Food,4293.0,0.503873
Merch,1771.5,0.207923
Drink,1554.0,0.182394
RainGear,901.5,0.105810


### Q3 — Which vendor has the highest *average* order revenue?

Report the average alongside the order count for each vendor. A high average on twelve orders is a different claim from a high average on two hundred.

In [4]:
# TODO
by_vendor = df.groupby('vendor_id').agg(
    order_count=('vendor_id', 'size'),
    qty=('qty', 'sum'),
    revenue=('revenue', 'sum')
)
by_vendor['avg_revenue'] = by_vendor['revenue'] / by_vendor['order_count']
by_vendor.sort_values('avg_revenue', ascending=False)
# This means V-01 has the highest average order value ($22.60)
# despite having fewer orders than V-18 or V-10, so its lead
# isn't just a volume effect.

,order_count,qty,revenue,avg_revenue
vendor_id,,,,
V-01,94,188,2124.0,22.595745
V-18,108,217,2349.0,21.750000
V-05,93,178,1914.0,20.580645
V-10,105,200,2133.0,20.314286


### Q4 — What share of revenue comes from Merch?

Print it as a percentage rounded to one decimal.

In [5]:
# TODO
merch_share = by_category.loc['Merch', 'revenue'] / by_category['revenue'].sum() * 100
print(f"Merch makes up {merch_share:.1f}% of total revenue.")
# This means as printed, that Merch makes up that much of the revenue.

Merch makes up 20.8% of total revenue.


### Q5 — Join in the vendor names.

The frame only has `vendor_id`. Merge the lookup below so your report is readable.

**Requirements:** left join, `validate='many_to_one'`, and prove the row count and revenue total did not change. One vendor id in the orders is not in this lookup — find it, and decide what to do about it.

In [6]:
vendor_names = pd.DataFrame({
    'vendor_id': ['V-01', 'V-05', 'V-10'],
    'vendor_name': ['Hoos Burgers', 'Rotunda Tacos', 'Cav Merch North'],
})

# TODO: merge, validate, and report the unmatched vendor
joined = df.merge(vendor_names, on='vendor_id', how='left', validate='many_to_one')

assert len(joined) == len(df), 'merge changed row count'
assert abs(joined['revenue'].sum() - df['revenue'].sum()) < 0.01, 'merge changed revenue'

missing = joined[joined['vendor_name'].isna()]
print(f"unmatched vendor_id: {missing['vendor_id'].unique()}, "
      f"{len(missing)} orders, ${missing['revenue'].sum():.2f} at stake")

joined['vendor_name'] = joined['vendor_name'].fillna('Unknown vendor (V-18)')

joined

unmatched vendor_id: ['V-18'], 108 orders, $2349.00 at stake


,vendor_id,category,qty,price,revenue,vendor_name
0,V-10,Drink,2,24.0,48.0,Cav Merch North
1,V-18,RainGear,1,12.0,12.0,Unknown vendor (V-18)
2,V-18,Drink,3,4.5,13.5,Unknown vendor (V-18)
3,V-10,Food,2,12.0,24.0,Cav Merch North
4,V-18,Drink,3,7.5,22.5,Unknown vendor (V-18)
...,...,...,...,...,...,...
395,V-18,Merch,1,12.0,12.0,Unknown vendor (V-18)
396,V-01,Merch,2,24.0,48.0,Hoos Burgers
397,V-10,Food,3,7.5,22.5,Cav Merch North
398,V-18,Merch,2,24.0,48.0,Unknown vendor (V-18)


**The unmatched vendor, and what I did about it:** V-18 has 108 orders and $2,349 in revenue (over a quarter of the night's total) but it's missing from the lookup table entirely. Dropping that many orders would understate the report, and leaving vendor_name as NaN would break the later pivot table (every V-18 row would disappear from a vendor_name-indexed pivot instead of showing up as its own line). I labeled it 'Unknown vendor (V-18)' and kept every row, so the revenue stays accounted for and the pivot table still shows exactly how much this mystery vendor sold.

### Q6 — A pivot table: vendors down the side, categories across the top, revenue in the cells.

Add row and column totals so it reads as a report rather than a grid of numbers.

In [7]:
# TODO
pd.pivot_table(joined, index='vendor_name', columns='category', values='revenue', aggfunc='sum', margins=True)
# This means Hoos Burgers earns disproportionately from
# Food (1,338 of its 2,124 total), while Cav Merch North's
# revenue is spread more evenly across categories

category,Drink,Food,Merch,RainGear,All
vendor_name,,,,,
Cav Merch North,502.5,1054.5,400.5,175.5,2133.0
Hoos Burgers,171.0,1338.0,373.5,241.5,2124.0
Rotunda Tacos,298.5,882.0,489.0,244.5,1914.0
Unknown vendor (V-18),582.0,1018.5,508.5,240.0,2349.0
All,1554.0,4293.0,1771.5,901.5,8520.0


### Q7 — Validate your work

**TODO:** uncomment and make these pass. Assign your results to the named variables as you go.

In [8]:
assert len(df) == 400
assert 8000 < df['revenue'].sum() < 9000, df['revenue'].sum()
assert abs(by_category['revenue'].sum() - df['revenue'].sum()) < 0.01
assert len(joined) == len(df), 'the vendor merge changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** What would you tell these vendors to do differently next game? One paragraph, with at least two numbers from your report in it.

**b)** Which of your seven answers is the least trustworthy, and why? Point at a specific weakness — a small group size, an unmatched vendor, a category that is really two things.

a - Food is completely carrying, where it's 50.4% of total revenue (\$4,293 of \$8,520) even though it's only 1/4 categories, so vendors should prioritize selling Food items and they should make sure they are not low stock during prime hours. RainGear in contrast, makes up the lowest 10.6% in revenue (\$901.50 of \$8,520), which may be worth watching depending on what the weather is that day (so don't over order on dry days but watch for rainy ones).

b - I am most hesitant about Q3. The average of V-01's order revenue (\$22.60) is the highest, but that's only from 94 orders (smallest of the 4 vendors), whereas V-10 has 105 orders and a lower average (\$20.31). Some unusually large orders might be bringing up V-01's average without it showing a real pattern for how the vendor sells. But, with sample sizes this close (94-108 orders), I wouldn't confidentaly rank vendor performance by average alone, I would want to look at the order-size distribution, not just the mean.